# COVID-19 Tweet Sentiment

Sentiment classification of ~45,000 COVID-19 tweets into 5 classes (Extremely Negative, Negative, Neutral, Positive, Extremely Positive). The train and test files are combined and re-split 70/30, and Multinomial Naive Bayes, Logistic Regression and Random Forest are compared on TF-IDF features.

**Data:** `Corona_NLP_train.csv` and `Corona_NLP_test.csv` (see [`../data/README.md`](../data/README.md)).

In [1]:
# Import the libraries
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import pickle
import matplotlib.pyplot as plt

DATA_DIR = "../data/"

In [3]:
# Load the train and test files
df1 = pd.read_csv(DATA_DIR + 'Corona_NLP_train.csv', encoding='latin-1', header=None)
df2 = pd.read_csv(DATA_DIR + 'Corona_NLP_test.csv', encoding='latin-1', header=None)

In [4]:
df1.head()

,0,1,2,3,4,5
0,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
1,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
2,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
3,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
4,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive


In [5]:
df2.head()

,0,1,2,3,4,5
0,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
1,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
2,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive
3,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive
4,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative


In [6]:
# Stack both files into a single DataFrame
df = pd.concat([df1, df2], ignore_index=True)

In [7]:
df

,0,1,2,3,4,5
0,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
1,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
2,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
3,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
4,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
...,...,...,...,...,...,...
44952,3794,48746,Israel ??,16-03-2020,Meanwhile In A Supermarket in Israel -- People...,Positive
44953,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative
44954,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral
44955,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative


In [8]:
# Use the first row as column names
df.columns = df.iloc[0]

In [9]:
# Drop the first row
df = df[1:].reset_index(drop=True)

In [10]:
df

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative
...,...,...,...,...,...,...
44951,3794,48746,Israel ??,16-03-2020,Meanwhile In A Supermarket in Israel -- People...,Positive
44952,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative
44953,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral
44954,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative


In [11]:
# Keep the tweet text and its sentiment
df = df[['OriginalTweet', 'Sentiment']]

In [12]:
df

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,advice Talk to your neighbours family to excha...,Positive
2,Coronavirus Australia: Woolworths to give elde...,Positive
3,My food stock is not the only one which is emp...,Positive
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative
...,...,...
44951,Meanwhile In A Supermarket in Israel -- People...,Positive
44952,Did you panic buy a lot of non-perishable item...,Negative
44953,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral
44954,Gov need to do somethings instead of biar je r...,Extremely Negative


In [13]:
# Convert categorical data to numerical data using replace
df['Sentiment'] = df['Sentiment'].replace({'Extremely Negative': 4, 'Negative': 3, 'Neutral': 2, 'Positive': 1, 'Extremely Positive': 0})
df['sentiment'] = df['Sentiment'].apply(lambda x: 0 if (x == 0 or x == 1) else (2 if (x == 3 or x == 4) else (1 if x == 2 else x)))

In [14]:
print(df['Sentiment'].unique())

[2 1 4 3 0 'Sentiment']


In [15]:
# Remove rows with unexpected values in 'Sentiment' (the second file's header row)
df = df[df['Sentiment'] != 'Sentiment']

In [17]:
# Number of tweets per sentiment class
print(df['Sentiment'].value_counts())

Sentiment
1    12369
3    10958
2     8332
0     7223
4     6073
Name: count, dtype: int64


In [18]:
# Step 2: Text preprocessing with CountVectorizer
# Bag-of-Words representation
df['OriginalTweet'] = df['OriginalTweet'].astype(str).str.lower()
vectorizer = CountVectorizer(stop_words='english', max_df=0.8, min_df=10, max_features=15000)
X = vectorizer.fit_transform(df['OriginalTweet'])
y = df['Sentiment']

In [19]:
# Step 2b: TF-IDF representation (the one used to train the models)
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df['OriginalTweet'])
y = df['Sentiment']

In [20]:
# Step 3: Split into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [21]:
# Convert the labels to integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Check the type and the unique values
print(y_train.dtype)
print(set(y_train))

int64
{0, 1, 2, 3, 4}


In [22]:
# Models to train
modelos = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
}

In [23]:
# Dictionary to store the results
resultados = {'Modelo': [], 'Precisión': [], 'F1-Score': [], 'Recall': [], 'Precision Avg.': []}

# Train and evaluate each model
for nombre, modelo in modelos.items():
    print(f"Evaluando modelo: {nombre}")

    # Train the model
    modelo.fit(X_train, y_train)

    # Predict on the test set
    y_pred = modelo.predict(X_test)

    # Compute the metrics (accuracy is stored as 'Precisión')
    precision = accuracy_score(y_test, y_pred)
    reporte = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    # Weighted-average metrics
    f1_score = reporte['weighted avg']['f1-score']
    recall = reporte['weighted avg']['recall']
    precision_avg = reporte['weighted avg']['precision']

    # Print the results
    print(f"\nPrecisión de {nombre}: {precision}")
    print("Matriz de Confusión:")
    print(confusion_matrix(y_test, y_pred))
    print("Reporte de Clasificación:")
    print(classification_report(y_test, y_pred))
    print("\n" + "="*60 + "\n")

    # Save the results
    resultados['Modelo'].append(nombre)
    resultados['Precisión'].append(precision)
    resultados['F1-Score'].append(f1_score)
    resultados['Recall'].append(recall)
    resultados['Precision Avg.'].append(precision_avg)

Evaluando modelo: Multinomial Naive Bayes

Precisión de Multinomial Naive Bayes: 0.4562912434195892
Matriz de Confusión:
[[ 514 1500   43  124    4]
 [ 181 2574  246  643   23]
 [  27  947  974  543   17]
 [  33 1181  259 1728  118]
 [   3  293   44 1104  364]]
Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.68      0.24      0.35      2185
           1       0.40      0.70      0.51      3667
           2       0.62      0.39      0.48      2508
           3       0.42      0.52      0.46      3319
           4       0.69      0.20      0.31      1808

    accuracy                           0.46     13487
   macro avg       0.56      0.41      0.42     13487
weighted avg       0.53      0.46      0.44     13487



Evaluando modelo: Logistic Regression

Precisión de Logistic Regression: 0.5602431971528138
Matriz de Confusión:
[[1127  904   57   79   18]
 [ 384 2209  469  535   70]
 [  26  431 1687  342   22]
 [  66  686  473 1697  3

In [24]:
# Step 5: Save the last trained model and the vectorizer for later use
with open('modelo_sentimiento_covid.pkl', 'wb') as f:
    pickle.dump(modelo, f)
with open('vectorizer_covid.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

## Results

| Model | Accuracy (5 classes) | Weighted F1 |
|---|---|---|
| Logistic Regression | **56.0%** | 0.56 |
| Random Forest | 51.5% | 0.51 |
| Multinomial Naive Bayes | 45.6% | 0.44 |

Label encoding: 0 = Extremely Positive, 1 = Positive, 2 = Neutral, 3 = Negative, 4 = Extremely Negative.

With 5 fine-grained classes, the task is much harder than binary sentiment. The largest errors are between an extreme class and its milder neighbour: with Logistic Regression, 904 *Extremely Positive* tweets were predicted as *Positive* and 758 *Extremely Negative* tweets as *Negative*. Logistic Regression is again the strongest model.